# MNIST Full SpikeEngine CUDA Scratchpad

This notebook is the `mnist_echo_state_scratchpad` experiment rebuilt on the real `SpikeEngineCUDA` path. The reservoir is a full spiking engine on a square torus topology, with CUDA kernels, low-rank weights, and k2-tree construction enabled. The only supervised parameters are the linear readouts.

The feature path uses membrane potential plus a spike-recency trace sampled from the engine state. That keeps the readout close to the LIF CartPole proof of concept while avoiding a separate ad hoc reservoir implementation.

In [ ]:
import gzip
import importlib
import math
import struct
import time
from dataclasses import dataclass
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import cupy as cp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

import spike_engine_cuda
import weights_cuda
import topologies

importlib.reload(weights_cuda)
importlib.reload(spike_engine_cuda)

from spike_engine_cuda import SpikeEngineCUDA
from topologies import square_torus

print(f"numpy {np.__version__}; cupy {cp.__version__}")

## MNIST Loader

The loader is dependency-light: it downloads the raw IDX files once into `.spikecore.cache/mnist` and parses them directly.

In [ ]:
MNIST_CACHE = Path(".spikecore.cache/mnist")
MNIST_BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
MNIST_FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}


def ensure_mnist(cache_dir: Path = MNIST_CACHE):
    cache_dir.mkdir(parents=True, exist_ok=True)
    for filename in MNIST_FILES.values():
        path = cache_dir / filename
        if not path.exists():
            print(f"downloading {filename}")
            urlretrieve(MNIST_BASE_URL + filename, path)


def read_mnist_images(path: Path):
    with gzip.open(path, "rb") as f:
        magic, count, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"bad image file magic {magic}: {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(count, rows, cols)


def read_mnist_labels(path: Path):
    with gzip.open(path, "rb") as f:
        magic, count = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"bad label file magic {magic}: {path}")
        return np.frombuffer(f.read(), dtype=np.uint8)


def load_mnist(cache_dir: Path = MNIST_CACHE):
    ensure_mnist(cache_dir)
    return (
        read_mnist_images(cache_dir / MNIST_FILES["train_images"]),
        read_mnist_labels(cache_dir / MNIST_FILES["train_labels"]),
        read_mnist_images(cache_dir / MNIST_FILES["test_images"]),
        read_mnist_labels(cache_dir / MNIST_FILES["test_labels"]),
    )


def select_balanced(images, labels, per_digit: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    indices = []
    for digit in range(10):
        digit_idx = np.flatnonzero(labels == digit)
        indices.extend(rng.choice(digit_idx, size=per_digit, replace=False).tolist())
    rng.shuffle(indices)
    return images[indices], labels[indices]


def sample_random_labeled(images, labels, count: int, rng):
    idx = rng.integers(0, len(labels), size=count)
    return images[idx], labels[idx]


def input_from_images(images):
    x = images[:, ::2, ::2].astype(np.float32) / 255.0
    return x.reshape(len(images), -1)


def one_hot(labels, classes: int = 10):
    y = np.zeros((len(labels), classes), dtype=np.float32)
    y[np.arange(len(labels)), np.asarray(labels, dtype=np.int64)] = 1.0
    return y

train_images_all, train_labels_all, test_images_all, test_labels_all = load_mnist()
print(train_images_all.shape, test_images_all.shape)

## Engine Configuration

The 14x14 MNIST input is mapped into a 28x28 square-torus spike engine. Each low-resolution pixel drives the center neuron of its corresponding 2x2 reservoir block. Recurrent weights are held near the single-step bifurcation estimate by using the engine's constant-weight mode; set `freeze_learning=False` to allow the engine STDP path to update weights.

In [ ]:
@dataclass
class EngineMNISTConfig:
    input_shape: tuple[int, int] = (14, 14)
    reservoir_side: int = 28
    rank: int = 64
    resting_mp: float = 0.1
    spike_threshold: float = 1.0
    decay_rate: float = 0.22
    recurrent_scale: float = 1.03
    freeze_learning: bool = True
    input_gain: float = 1.15
    pre_steps: int = 3
    on_steps: int = 32
    off_steps: int = 8
    readout_start: int = 12
    readout_stride: int = 4
    spike_tau: float = 10.0
    weight_init_scale: float = 0.01
    seed: int = 7
    use_k2tree: bool = True
    verify_k2tree: bool = False


@dataclass
class ProbeConfig:
    train_per_digit: int = 500
    test_per_digit: int = 100
    batch_size: int = 128
    ridge: float = 1e-2
    rls_delta: float = 1e-2
    rls_forgetting: float = 1.0
    rls_epochs: int = 1


ENGINE_CONFIG = EngineMNISTConfig()
PROBE_CONFIG = ProbeConfig()
print(ENGINE_CONFIG)


## SpikeEngine MNIST Reservoir

This wrapper owns no dynamics of its own. It only resets the engine, injects encoded input current into configured input neurons, and reads membrane/spike-recency features from the engine state.

In [ ]:
class SpikeEngineMNISTReservoir:
    frame_title = "SpikeEngine membrane potential"

    def __init__(self, config: EngineMNISTConfig):
        self.config = config
        self.rng = np.random.default_rng(config.seed)
        self.n_input = int(np.prod(config.input_shape))
        self.n_reservoir = config.reservoir_side * config.reservoir_side
        self.input_neurons = self._make_input_neurons()
        cp.random.seed(config.seed)
        self.engine = SpikeEngineCUDA(
            square_torus(config.reservoir_side),
            (config.reservoir_side, config.reservoir_side),
            rank=config.rank,
            resting_mp=config.resting_mp,
            decay_rate=config.decay_rate,
            weight_initializer=lambda size: cp.random.normal(0.0, config.weight_init_scale, size=size),
            use_k2tree=config.use_k2tree,
            verify_k2tree=config.verify_k2tree,
            verify_progress_every=2000,
        )
        self.engine.SPIKE_THRESHOLD = cp.float32(config.spike_threshold)
        self.engine.set_input_neurons(self.input_neurons)
        target, w_accum, w_instant = self.engine.set_constant_weights_near_bifurcation(
            input_period=1,
            scale=config.recurrent_scale,
            freeze_learning=config.freeze_learning,
        )
        self.weight_summary = {"target": target, "w_accum": w_accum, "w_instant": w_instant}
        self.reset()

    @property
    def n_features(self):
        return 2 * self.n_reservoir

    def _make_input_neurons(self):
        side = self.config.reservoir_side
        rows, cols = self.config.input_shape
        if side != rows * 2 or side != cols * 2:
            raise ValueError("default mapping expects a 2x square scale-up from input to reservoir")
        ids = []
        for r in range(rows):
            for c in range(cols):
                ids.append((2 * r + 1) * side + (2 * c + 1))
        return cp.asarray(ids, dtype=cp.int32)

    def reset(self):
        self.engine.reset_state()
        self.tick = 2

    def apply_runtime_controls(
        self,
        *,
        input_gain: float | None = None,
        recurrent_scale: float | None = None,
        decay_rate: float | None = None,
        spike_threshold: float | None = None,
        resting_mp: float | None = None,
        spike_tau: float | None = None,
        reset_state: bool = False,
    ):
        cfg = self.config
        if input_gain is not None:
            cfg.input_gain = float(input_gain)
        if decay_rate is not None:
            cfg.decay_rate = float(decay_rate)
            self.engine.DECAY_RATE = cp.float32(cfg.decay_rate)
        if spike_threshold is not None:
            cfg.spike_threshold = float(spike_threshold)
            self.engine.SPIKE_THRESHOLD = cp.float32(cfg.spike_threshold)
        if resting_mp is not None:
            old_rest = float(self.engine.RESTING_MP)
            cfg.resting_mp = float(resting_mp)
            self.engine.RESTING_MP = cp.float32(cfg.resting_mp)
            self.engine.membrane_potentials += cp.float32(cfg.resting_mp - old_rest)
        if spike_tau is not None:
            cfg.spike_tau = float(spike_tau)
        if recurrent_scale is not None:
            cfg.recurrent_scale = float(recurrent_scale)
            target, w_accum, w_instant = self.engine.set_constant_weights_near_bifurcation(
                input_period=1,
                scale=cfg.recurrent_scale,
                freeze_learning=cfg.freeze_learning,
            )
            self.weight_summary = {"target": target, "w_accum": w_accum, "w_instant": w_instant}
        if reset_state:
            self.reset()
        return self.weight_summary

    def envelope(self, step: int):
        cfg = self.config
        if step < cfg.pre_steps:
            return 0.0
        if step < cfg.pre_steps + cfg.on_steps:
            return 1.0
        return 0.0

    def step(self, input_vector):
        cfg = self.config
        values = np.asarray(input_vector, dtype=np.float32) * (cfg.input_gain * self.envelope(self.tick))
        self.engine.advance_static_input(values, self.tick, self.input_neurons, full_decay=False)
        self.tick += 1

    def _snapshot_feature(self):
        self.engine._decay_all(self.tick)
        mp = cp.asnumpy(self.engine.membrane_potentials).astype(np.float32)
        last = cp.asnumpy(self.engine.last_spiked).astype(np.int32)
        age = np.maximum(0, self.tick - last).astype(np.float32)
        trace = np.exp(-age / max(1e-6, self.config.spike_tau)).astype(np.float32)
        trace[last <= 0] = 0.0
        mp_feature = (mp - self.config.resting_mp).astype(np.float32)
        return np.concatenate([mp_feature, trace]).astype(np.float32)

    def collect_features(self, input_vectors, batch_size: int = 128, progress_desc: str = "features"):
        cfg = self.config
        input_vectors = np.asarray(input_vectors, dtype=np.float32)
        total_steps = cfg.pre_steps + cfg.on_steps + cfg.off_steps
        out = np.empty((len(input_vectors), self.n_features), dtype=np.float32)
        for start in tqdm(range(0, len(input_vectors), batch_size), desc=progress_desc):
            stop = min(start + batch_size, len(input_vectors))
            for row, x in enumerate(input_vectors[start:stop], start=start):
                self.reset()
                accum = np.zeros(self.n_features, dtype=np.float32)
                count = 0
                for _ in range(total_steps):
                    self.step(x)
                    if self.tick >= cfg.readout_start and (self.tick - cfg.readout_start) % cfg.readout_stride == 0:
                        accum += self._snapshot_feature()
                        count += 1
                out[row] = accum / max(1, count)
        return out

    def current_frame(self):
        self.engine._decay_all(self.tick)
        return cp.asnumpy(self.engine.membrane_potentials).reshape(self.config.reservoir_side, self.config.reservoir_side)

    def stats(self):
        frame = self.current_frame()
        last = cp.asnumpy(self.engine.last_spiked).astype(np.int32)
        recent = np.mean((self.tick - last) <= 1)
        ever = np.mean(last > 0)
        return {
            "mp_mean": float(frame.mean()),
            "mp_std": float(frame.std()),
            "mp_max": float(frame.max()),
            "recent_spike_fraction": float(recent),
            "ever_spiked_fraction": float(ever),
        }


reservoir = SpikeEngineMNISTReservoir(ENGINE_CONFIG)
print("input neurons:", reservoir.n_input, "reservoir neurons:", reservoir.n_reservoir, "features:", reservoir.n_features)
print("weight summary:", reservoir.weight_summary)
print("k2tree constructed:", reservoir.engine.weights.k2tree is not None)


## Linear Readouts

The ridge probe is the closed-form baseline. The RLS probe is the online readout used for incremental training.

In [ ]:
class RidgeProbe:
    def __init__(self, weights, mean, std):
        self.weights = weights.astype(np.float32)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        self.readout_name = "batch ridge probe"

    @classmethod
    def fit(cls, features, labels, ridge: float = 1e-2):
        X = np.asarray(features, dtype=np.float32)
        mean = X.mean(axis=0)
        std = X.std(axis=0) + 1e-6
        Xn = (X - mean) / std
        X_aug = np.column_stack([Xn, np.ones(len(Xn), dtype=np.float32)])
        Y = one_hot(labels)
        lhs = X_aug.T @ X_aug + ridge * np.eye(X_aug.shape[1], dtype=np.float32)
        rhs = X_aug.T @ Y
        return cls(np.linalg.solve(lhs, rhs), mean, std)

    def logits(self, features):
        X = np.asarray(features, dtype=np.float32)
        Xn = (X - self.mean) / self.std
        X_aug = np.column_stack([Xn, np.ones(len(Xn), dtype=np.float32)])
        return X_aug @ self.weights

    def probabilities(self, features):
        z = self.logits(features)
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / np.maximum(1e-8, e.sum(axis=1, keepdims=True))

    def predict(self, features):
        return self.logits(features).argmax(axis=1)


class OnlineRLSProbe:
    def __init__(self, n_features: int, delta: float = 1e-2, forgetting: float = 1.0):
        self.n_features = int(n_features)
        self.delta = float(delta)
        self.forgetting = float(forgetting)
        self.mean = np.zeros(self.n_features, dtype=np.float32)
        self.std = np.ones(self.n_features, dtype=np.float32)
        self.weights = np.zeros((self.n_features + 1, 10), dtype=np.float32)
        self.P = np.eye(self.n_features + 1, dtype=np.float32) / max(1e-8, self.delta)
        self.readout_name = "online RLS probe"

    def reset_weights(self, delta: float | None = None):
        if delta is not None:
            self.delta = float(delta)
        self.weights = np.zeros((self.n_features + 1, 10), dtype=np.float32)
        self.P = np.eye(self.n_features + 1, dtype=np.float32) / max(1e-8, self.delta)

    def set_normalizer(self, features):
        X = np.asarray(features, dtype=np.float32)
        self.mean = X.mean(axis=0).astype(np.float32)
        self.std = (X.std(axis=0) + 1e-6).astype(np.float32)

    def _augment(self, features):
        X = (np.asarray(features, dtype=np.float32) - self.mean) / self.std
        return np.column_stack([X, np.ones(len(X), dtype=np.float32)])

    def logits(self, features):
        return self._augment(features) @ self.weights

    def probabilities(self, features):
        z = self.logits(features)
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / np.maximum(1e-8, e.sum(axis=1, keepdims=True))

    def predict(self, features):
        return self.logits(features).argmax(axis=1)

    def partial_fit(self, features, labels, forgetting: float | None = None):
        lam = self.forgetting if forgetting is None else float(forgetting)
        X = self._augment(features)
        Y = one_hot(labels)
        P = self.P / max(1e-8, lam)
        S = np.eye(len(X), dtype=np.float32) + X @ P @ X.T
        K = P @ X.T @ np.linalg.inv(S)
        self.weights += K @ (Y - X @ self.weights)
        self.P = P - K @ X @ P

    def train_epochs(self, features, labels, epochs: int = 1, batch_size: int = 128, seed: int = 0):
        rng = np.random.default_rng(seed)
        idx = np.arange(len(labels))
        for epoch in range(epochs):
            rng.shuffle(idx)
            for start in tqdm(range(0, len(idx), batch_size), desc=f"RLS epoch {epoch + 1}/{epochs}"):
                batch_idx = idx[start:start + batch_size]
                self.partial_fit(features[batch_idx], labels[batch_idx])


def accuracy(probe, features, labels):
    return float(np.mean(probe.predict(features) == np.asarray(labels)))

## Feature Extraction And Baselines

The defaults match the earlier MNIST scratchpad scale. Lower `train_per_digit` for quick iteration; raise it when you want a stronger readout estimate.

In [ ]:
train_images, train_labels = select_balanced(train_images_all, train_labels_all, PROBE_CONFIG.train_per_digit, seed=1)
test_images, test_labels = select_balanced(test_images_all, test_labels_all, PROBE_CONFIG.test_per_digit, seed=2)

X_train_input = input_from_images(train_images)
X_test_input = input_from_images(test_images)

start = time.perf_counter()
train_features = reservoir.collect_features(X_train_input, batch_size=PROBE_CONFIG.batch_size, progress_desc="train features")
test_features = reservoir.collect_features(X_test_input, batch_size=PROBE_CONFIG.batch_size, progress_desc="test features")
print(f"feature extraction seconds: {time.perf_counter() - start:.1f}")
print("feature stats:", train_features.mean(), train_features.std(), train_features.min(), train_features.max())

ridge_probe = RidgeProbe.fit(train_features, train_labels, ridge=PROBE_CONFIG.ridge)
print(f"ridge train/test: {accuracy(ridge_probe, train_features, train_labels):.3f} / {accuracy(ridge_probe, test_features, test_labels):.3f}")

rls_probe = OnlineRLSProbe(reservoir.n_features, delta=PROBE_CONFIG.rls_delta, forgetting=PROBE_CONFIG.rls_forgetting)
rls_probe.set_normalizer(train_features)
rls_probe.train_epochs(train_features, train_labels, epochs=PROBE_CONFIG.rls_epochs, batch_size=PROBE_CONFIG.batch_size, seed=3)
print(f"RLS train/test: {accuracy(rls_probe, train_features, train_labels):.3f} / {accuracy(rls_probe, test_features, test_labels):.3f}")

## Confusion Matrices

In [ ]:
def confusion_matrix_counts(labels, predictions, classes: int = 10):
    matrix = np.zeros((classes, classes), dtype=np.int64)
    for y, yhat in zip(labels, predictions):
        matrix[int(y), int(yhat)] += 1
    return matrix


def plot_confusion_matrices(probe=rls_probe):
    train_cm = confusion_matrix_counts(train_labels, probe.predict(train_features))
    test_cm = confusion_matrix_counts(test_labels, probe.predict(test_features))
    fig = make_subplots(rows=1, cols=2, subplot_titles=("train", "test"))
    fig.add_trace(go.Heatmap(z=train_cm, colorscale="Blues", showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=test_cm, colorscale="Blues", showscale=True), row=1, col=2)
    fig.update_layout(width=850, height=380, title=f"Confusion matrices: {probe.readout_name}")
    fig.update_xaxes(title="predicted")
    fig.update_yaxes(title="label", autorange="reversed")
    fig.show()
    return train_cm, test_cm

train_confusion, test_confusion = plot_confusion_matrices(rls_probe)

## Single Digit Probe

Use this cell to inspect the engine response and readout probabilities for individual images.

In [ ]:
def run_digit_example(index: int = 0, source: str = "test", probe=rls_probe):
    images, labels = (test_images_all, test_labels_all) if source == "test" else (train_images_all, train_labels_all)
    image = images[index]
    label = int(labels[index])
    x = input_from_images(image[None, ...])[0]
    feature = reservoir.collect_features(x[None, ...], batch_size=1, progress_desc="single digit")
    probs = probe.probabilities(feature)[0]
    frame = reservoir.current_frame()

    fig = make_subplots(rows=1, cols=3, subplot_titles=(f"image label={label}", "engine membrane", "readout"), specs=[[{}, {}, {"type": "bar"}]])
    fig.add_trace(go.Heatmap(z=image, colorscale="Gray", showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=frame, colorscale="Viridis", showscale=False), row=1, col=2)
    fig.add_trace(go.Bar(x=list(range(10)), y=probs), row=1, col=3)
    fig.update_layout(width=1050, height=360, title=f"prediction={int(np.argmax(probs))} confidence={float(np.max(probs)):.3f}")
    fig.show()
    return probs

# probs = run_digit_example(0, "test")

## Live Online Learning Controls

This is the full-engine replacement for the live MNIST controls in the echo-state scratchpad. Digit buttons present examples through `SpikeEngineCUDA`; batch and auto-train controls update the online RLS readout from fresh MNIST samples. Engine parameters that affect presentation can be adjusted without rebuilding the engine.

In [ ]:
try:
    import threading
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    widgets = None
    print(f"ipywidgets unavailable: {exc}")


class LiveMNISTSpikeEngineView:
    def __init__(self, reservoir: SpikeEngineMNISTReservoir, probe):
        if widgets is None:
            raise RuntimeError("ipywidgets is required for the live view")
        self.reservoir = reservoir
        self.probe = probe
        self.rng = np.random.default_rng(123)
        self._stop = threading.Event()
        self._train_stop = threading.Event()
        self._thread = None
        self._train_thread = None
        self._lock = threading.RLock()
        self.current_digit = None
        self.current_image = np.zeros((28, 28), dtype=np.uint8)
        self.current_input = np.zeros(self.reservoir.n_input, dtype=np.float32)
        self.readout_sum = np.zeros(self.reservoir.n_features, dtype=np.float32)
        self.readout_count = 0
        self.sequence_step = 0
        self.last_eval_acc = None
        self.last_batch_acc = None

        self.input_fig = go.FigureWidget(data=[go.Heatmap(z=np.zeros(self.reservoir.config.input_shape), colorscale="Gray", zmin=0, zmax=1, showscale=False)])
        self.input_fig.update_layout(width=310, height=310, margin=dict(l=0, r=0, b=0, t=24), title=dict(text="14x14 input", x=0.5, y=0.98, font=dict(size=13)), xaxis=dict(visible=False), yaxis=dict(visible=False, autorange="reversed"))
        self.reservoir_fig = go.FigureWidget(data=[go.Heatmap(z=np.zeros((self.reservoir.config.reservoir_side, self.reservoir.config.reservoir_side)), colorscale="Viridis", showscale=False)])
        self.reservoir_fig.update_layout(width=430, height=430, margin=dict(l=0, r=0, b=0, t=24), title=dict(text="SpikeEngine membrane", x=0.5, y=0.98, font=dict(size=13)), xaxis=dict(visible=False), yaxis=dict(visible=False, autorange="reversed"))
        self.probe_fig = go.FigureWidget(data=[go.Bar(x=list(range(10)), y=np.full(10, 0.1, dtype=np.float32), marker_color="#4c78a8")])
        self.probe_fig.update_layout(width=640, height=260, margin=dict(l=30, r=10, b=30, t=24), title=dict(text="readout probabilities", x=0.5, y=0.98, font=dict(size=13)), yaxis=dict(range=[0, 1]), xaxis=dict(dtick=1))

        self.digit_buttons = [widgets.Button(description=str(d), layout=widgets.Layout(width="42px")) for d in range(10)]
        for digit, button in enumerate(self.digit_buttons):
            button.on_click(lambda _, d=digit: self.present_digit(d))
        self.train_batch_button = widgets.Button(description="Train Batch", icon="graduation-cap", button_style="info")
        self.auto_train_button = widgets.Button(description="Auto Train", icon="play", button_style="success")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.reset_state_button = widgets.Button(description="Reset Engine", icon="refresh")
        self.reset_probe_button = widgets.Button(description="Reset Probe", icon="eraser")
        self.eval_button = widgets.Button(description="Eval", icon="check")
        self.apply_dynamics_button = widgets.Button(description="Apply Dynamics", icon="sliders", button_style="warning")
        self.train_batch_button.on_click(lambda _: self.train_one_batch_async())
        self.auto_train_button.on_click(lambda _: self.start_auto_train())
        self.stop_button.on_click(lambda _: self.stop_all())
        self.reset_state_button.on_click(lambda _: self.reset_state())
        self.reset_probe_button.on_click(lambda _: self.reset_probe())
        self.eval_button.on_click(lambda _: self.evaluate_async())
        self.apply_dynamics_button.on_click(lambda _: self.apply_dynamics())

        self.sample_source = widgets.Dropdown(options=["train", "test"], value="train", description="source")
        self.train_on_present = widgets.Checkbox(value=True, description="train on presented")
        self.reset_before_image = widgets.Checkbox(value=True, description="reset before image")
        self.reset_on_dynamics = widgets.Checkbox(value=True, description="reset on dynamics apply", style={"description_width": "initial"})
        cfg = self.reservoir.config
        slider_style = {"description_width": "initial"}
        self.input_gain = widgets.FloatSlider(value=cfg.input_gain, min=0.0, max=6.0, step=0.05, description="input gain", continuous_update=False, style=slider_style)
        self.recurrent_scale = widgets.FloatSlider(value=cfg.recurrent_scale, min=0.0, max=2.5, step=0.01, readout_format=".2f", description="recurrent scale", continuous_update=False, style=slider_style)
        self.decay_rate = widgets.FloatSlider(value=cfg.decay_rate, min=0.0, max=0.75, step=0.01, readout_format=".2f", description="decay", continuous_update=False, style=slider_style)
        self.spike_threshold = widgets.FloatSlider(value=cfg.spike_threshold, min=0.05, max=3.0, step=0.05, readout_format=".2f", description="threshold", continuous_update=False, style=slider_style)
        self.resting_mp = widgets.FloatSlider(value=cfg.resting_mp, min=-0.5, max=0.95, step=0.01, readout_format=".2f", description="resting MP", continuous_update=False, style=slider_style)
        self.spike_tau = widgets.FloatSlider(value=cfg.spike_tau, min=1.0, max=80.0, step=1.0, readout_format=".0f", description="spike trace tau", continuous_update=False, style=slider_style)
        self.pre_steps = widgets.IntSlider(value=cfg.pre_steps, min=0, max=50, description="pre", continuous_update=False)
        self.on_steps = widgets.IntSlider(value=cfg.on_steps, min=1, max=120, description="on", continuous_update=False)
        self.off_steps = widgets.IntSlider(value=cfg.off_steps, min=0, max=80, description="off", continuous_update=False)
        self.readout_start = widgets.IntSlider(value=cfg.readout_start, min=0, max=180, description="readout start", continuous_update=False, style=slider_style)
        self.readout_stride = widgets.IntSlider(value=cfg.readout_stride, min=1, max=20, description="readout stride", continuous_update=False, style=slider_style)
        self.steps_per_frame = widgets.IntSlider(value=1, min=1, max=20, description="steps/frame", continuous_update=False, style=slider_style)
        self.delay_ms = widgets.IntSlider(value=20, min=0, max=250, step=5, description="delay ms", continuous_update=False, style=slider_style)
        self.reservoir_zmax = widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05, description="mp zmax", continuous_update=False, style=slider_style)
        self.train_batch_size = widgets.IntSlider(value=64, min=1, max=2048, step=1, description="batch", continuous_update=False)
        self.auto_delay_ms = widgets.IntSlider(value=100, min=0, max=2000, step=25, description="train delay", continuous_update=False, style=slider_style)
        self.eval_per_digit = widgets.IntSlider(value=20, min=5, max=200, step=5, description="eval/digit", continuous_update=False, style=slider_style)
        self.rls_delta = widgets.FloatLogSlider(value=float(getattr(self.probe, "delta", PROBE_CONFIG.rls_delta)), base=10, min=-6, max=2, step=0.25, description="RLS delta", continuous_update=False, style=slider_style)
        self.rls_forgetting = widgets.FloatSlider(value=float(getattr(self.probe, "forgetting", PROBE_CONFIG.rls_forgetting)), min=0.90, max=1.0, step=0.001, readout_format=".3f", description="RLS forget", continuous_update=False, style=slider_style)
        self.status = widgets.HTML(value="idle")

    def _sync_config(self, reset_state: bool = False):
        cfg = self.reservoir.config
        self.reservoir.apply_runtime_controls(
            input_gain=float(self.input_gain.value),
            recurrent_scale=float(self.recurrent_scale.value),
            decay_rate=float(self.decay_rate.value),
            spike_threshold=float(self.spike_threshold.value),
            resting_mp=float(self.resting_mp.value),
            spike_tau=float(self.spike_tau.value),
            reset_state=reset_state,
        )
        cfg.pre_steps = int(self.pre_steps.value)
        cfg.on_steps = int(self.on_steps.value)
        cfg.off_steps = int(self.off_steps.value)
        cfg.readout_start = int(self.readout_start.value)
        cfg.readout_stride = int(self.readout_stride.value)
        if hasattr(self.probe, "forgetting"):
            self.probe.forgetting = float(self.rls_forgetting.value)

    def _sample_digit(self, digit: int):
        images, labels = (train_images_all, train_labels_all) if self.sample_source.value == "train" else (test_images_all, test_labels_all)
        choices = np.flatnonzero(labels == digit)
        idx = int(self.rng.choice(choices))
        image = images[idx]
        return image, input_from_images(image[None, :, :])[0]

    def _total_steps(self):
        return int(self.pre_steps.value + self.on_steps.value + self.off_steps.value)

    def _current_feature(self):
        if self.readout_count > 0:
            return self.readout_sum / self.readout_count
        return self.reservoir._snapshot_feature()

    def _draw(self, env_value=0.0):
        feature = self._current_feature()
        probs = self.probe.probabilities(feature[None, :])[0]
        pred = int(np.argmax(probs))
        input_grid = (self.current_input * env_value).reshape(self.reservoir.config.input_shape)
        reservoir_grid = self.reservoir.current_frame()
        last = cp.asnumpy(self.reservoir.engine.last_spiked).astype(np.int32)
        recent = float(np.mean((self.reservoir.tick - last) <= 1))
        ever = float(np.mean(last > 0))
        with self.input_fig.batch_update():
            self.input_fig.data[0].z = input_grid
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].z = reservoir_grid
            self.reservoir_fig.data[0].zmax = float(self.reservoir_zmax.value)
        with self.probe_fig.batch_update():
            self.probe_fig.data[0].y = probs
            colors = ["#4c78a8"] * 10
            colors[pred] = "#f58518"
            self.probe_fig.data[0].marker.color = colors
        true_text = "?" if self.current_digit is None else str(self.current_digit)
        eval_text = "n/a" if self.last_eval_acc is None else f"{self.last_eval_acc:.3f}"
        batch_text = "n/a" if self.last_batch_acc is None else f"{self.last_batch_acc:.3f}"
        self.status.value = (
            f"true={true_text} pred={pred} p={probs[pred]:.3f} step={self.sequence_step}/{self._total_steps()} "
            f"readout_n={self.readout_count} recent={recent:.3f} ever={ever:.3f} "
            f"mp_mean={reservoir_grid.mean():.3f} mp_max={reservoir_grid.max():.3f} batch_acc={batch_text} eval={eval_text}"
        )

    def apply_dynamics(self):
        self.stop_presentation()
        with self._lock:
            self._sync_config(reset_state=bool(self.reset_on_dynamics.value))
            self.readout_sum.fill(0.0)
            self.readout_count = 0
            self.sequence_step = 0
            self._draw(0.0)
            ws = self.reservoir.weight_summary
            self.status.value += f" | target_w={ws['target']:.4f}"

    def _run_sequence(self):
        self._sync_config()
        self.readout_sum.fill(0.0)
        self.readout_count = 0
        self.sequence_step = 0
        total = self._total_steps()
        while not self._stop.is_set() and self.sequence_step < total:
            with self._lock:
                env_value = 0.0
                for _ in range(int(self.steps_per_frame.value)):
                    if self.sequence_step >= total:
                        break
                    env_value = self.reservoir.envelope(self.sequence_step)
                    self.reservoir.step(self.current_input)
                    if self.reservoir.tick >= self.reservoir.config.readout_start and (self.reservoir.tick - self.reservoir.config.readout_start) % self.reservoir.config.readout_stride == 0:
                        self.readout_sum += self.reservoir._snapshot_feature()
                        self.readout_count += 1
                    self.sequence_step += 1
                self._draw(env_value)
            delay = int(self.delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)
        with self._lock:
            if self.current_digit is not None and self.train_on_present.value and self.readout_count > 0:
                self.probe.partial_fit(self._current_feature()[None, :], [self.current_digit])
            self._draw(0.0)

    def present_digit(self, digit: int):
        self.stop_presentation()
        with self._lock:
            self.current_digit = int(digit)
            self.current_image, self.current_input = self._sample_digit(digit)
            self._sync_config(reset_state=bool(self.reset_before_image.value))
        self._stop.clear()
        self._thread = threading.Thread(target=self._run_sequence, daemon=True, name="mnist-spikeengine-present")
        self._thread.start()

    def _training_features_from_random_batch(self):
        self._sync_config()
        images, labels = sample_random_labeled(train_images_all, train_labels_all, int(self.train_batch_size.value), self.rng)
        inputs = input_from_images(images)
        features = self.reservoir.collect_features(inputs, batch_size=min(128, int(self.train_batch_size.value)), progress_desc="live train features")
        return features, labels

    def train_one_batch(self):
        with self._lock:
            features, labels = self._training_features_from_random_batch()
            self.probe.partial_fit(features, labels)
            self.last_batch_acc = accuracy(self.probe, features, labels)
            self._draw(0.0)

    def train_one_batch_async(self):
        threading.Thread(target=self.train_one_batch, daemon=True, name="mnist-spikeengine-train-batch").start()

    def _auto_train_loop(self):
        while not self._train_stop.is_set():
            self.train_one_batch()
            delay = int(self.auto_delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)

    def start_auto_train(self):
        if self._train_thread is not None and self._train_thread.is_alive():
            return
        self._train_stop.clear()
        self._train_thread = threading.Thread(target=self._auto_train_loop, daemon=True, name="mnist-spikeengine-auto-train")
        self._train_thread.start()

    def evaluate(self):
        with self._lock:
            self._sync_config()
            images, labels = sample_random_labeled(test_images_all, test_labels_all, int(self.eval_per_digit.value) * 10, self.rng)
            features = self.reservoir.collect_features(input_from_images(images), batch_size=128, progress_desc="live eval features")
            self.last_eval_acc = accuracy(self.probe, features, labels)
            self._draw(0.0)

    def evaluate_async(self):
        threading.Thread(target=self.evaluate, daemon=True, name="mnist-spikeengine-eval").start()

    def stop_presentation(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)

    def stop_training(self):
        self._train_stop.set()
        if self._train_thread is not None:
            self._train_thread.join(timeout=2)

    def stop_all(self):
        self.stop_presentation()
        self.stop_training()

    def reset_state(self):
        self.stop_presentation()
        with self._lock:
            self.reservoir.reset()
            self.current_digit = None
            self.current_input.fill(0.0)
            self.readout_sum.fill(0.0)
            self.readout_count = 0
            self.sequence_step = 0
            self._draw(0.0)

    def reset_probe(self):
        with self._lock:
            self.probe.delta = float(self.rls_delta.value)
            self.probe.reset_weights()
            self.last_eval_acc = None
            self.last_batch_acc = None
            self._draw(0.0)

    def display(self):
        controls = widgets.VBox([
            widgets.HBox(self.digit_buttons + [self.train_batch_button, self.auto_train_button, self.stop_button]),
            widgets.HBox([self.reset_state_button, self.reset_probe_button, self.eval_button, self.apply_dynamics_button, self.sample_source, self.train_on_present, self.reset_before_image, self.reset_on_dynamics]),
            widgets.HBox([self.input_gain, self.recurrent_scale, self.decay_rate, self.spike_threshold]),
            widgets.HBox([self.resting_mp, self.spike_tau, self.pre_steps, self.on_steps, self.off_steps]),
            widgets.HBox([self.readout_start, self.readout_stride, self.steps_per_frame, self.delay_ms, self.reservoir_zmax]),
            widgets.HBox([self.train_batch_size, self.rls_delta, self.rls_forgetting, self.auto_delay_ms, self.eval_per_digit]),
            self.status,
        ])
        display(widgets.VBox([controls, widgets.HBox([self.input_fig, self.reservoir_fig]), self.probe_fig]))


try:
    live_mnist.stop_all()
except NameError:
    pass

live_mnist = LiveMNISTSpikeEngineView(reservoir, rls_probe)
live_mnist.display()


## Notes For Larger Runs

- `record_stride` is not used here; features are sampled directly from engine state.
- To inspect raw dynamics, use `engine.start_static_record(...)` with the memory-safe streaming path.
- The current feature extraction is intentionally conservative and synchronizes while reading features. The next performance step is a CUDA feature-sampling kernel that accumulates readout features on device and transfers one vector per sample.